# 01 -- Preprocessing (Step 1)Loads BFRD, cleans it, deduplicates, and saves `cleaned_bfrd.csv` to Google Drive so every later notebook can reuse it without re-downloading.

## Mount Google Drive (persists files across all your notebooks)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/BanglaFakeReviewDetector'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
!pip install -q datasets nltk

In [ ]:
import re
import nltk
import pandas as pd
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
print("Setup complete.")

## Preprocessing functions

In [ ]:
BANGLA_STOPWORDS = set(stopwords.words('bengali'))
_porter = PorterStemmer()

EMOJI_PATTERN = re.compile(
    "[" "\U0001F300-\U0001FAFF" "\U00002600-\U000027BF"
    "\U0001F1E6-\U0001F1FF" "\U00002190-\U000021FF" "]+", flags=re.UNICODE)

def remove_urls(text):
    return re.sub(r"http\S+|www\.\S+", " ", text)

def remove_emojis(text):
    return EMOJI_PATTERN.sub(" ", text)

def normalize_repeated_punctuation(text):
    text = re.sub(r"([\u0964!?.\-_=~*])\1{1,}", r"\1", text)
    text = re.sub(r"(#\w+)(\s*\1){1,}", r"\1", text)
    return text

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = remove_urls(text)
    text = remove_emojis(text)
    text = normalize_repeated_punctuation(text)
    return re.sub(r"\s+", " ", text).strip()

def is_bangla_token(token):
    if not token:
        return False
    bangla_chars = sum(1 for ch in token if "\u0980" <= ch <= "\u09FF")
    return bangla_chars > len(token) / 2

def basic_tokenize(text):
    return re.findall(r"[\u0980-\u09FFa-zA-Z0-9]+", text)

def remove_stopwords(tokens):
    return [t for t in tokens if t not in BANGLA_STOPWORDS]

def stem_latin_tokens(tokens):
    return [_porter.stem(t) if not is_bangla_token(t) else t for t in tokens]

def preprocess_review(raw_text):
    cleaned = clean_text(raw_text)
    tokens = basic_tokenize(cleaned)
    filtered = stem_latin_tokens(remove_stopwords(tokens))
    return {"cleaned_text": cleaned, "tokens": filtered}

print(f"Loaded {len(BANGLA_STOPWORDS)} Bengali stopwords.")

## Load, clean, deduplicate, save

In [ ]:
from datasets import load_dataset

ds = load_dataset("shawon95/Bengali-Fake-Review-Dataset")
df = ds["train"].to_pandas()
df = df.rename(columns={c: "text" for c in df.columns if c.lower() in ("review", "text")})
df = df.rename(columns={c: "label" for c in df.columns if c.lower() in ("label", "class")})

processed = df["text"].apply(preprocess_review)
df["cleaned_text"] = processed.apply(lambda d: d["cleaned_text"])
df["is_code_mixed"] = df["text"].apply(lambda t: any(c.isalpha() and ord(c) < 128 for c in t) and any('\u0980' <= c <= '\u09FF' for c in t))
before = len(df)
df = df[df["cleaned_text"].str.len() > 0].drop_duplicates(subset=["cleaned_text"]).reset_index(drop=True)
print(f"Removed {before - len(df)} empty/duplicate reviews ({before} -> {len(df)}).")

counts = df["label"].value_counts()
print("Label counts:\n", counts)

df.to_csv("cleaned_bfrd.csv", index=False, encoding="utf-8-sig")
print("\nSaved cleaned_bfrd.csv to Google Drive -- ready for the next notebook.")